# **📘 M2C2 DataKit Notebook: Universal Loading, Assurance, and Scoring**

This notebook demonstrates a full analytic pipeline using the `m2datakit` package. It uses the `LASSIE` class to load, validate, score, and optionally export data from multiple source types.

**Version Date: July 28th, 2026**


---

## 🎯 Purpose

Enable researchers to plug in data from varied sources (e.g., MongoDB, UAS, MetricWire, CSV bundles) and apply a consistent pipeline for:

- Input validation

- Scoring via predefined rules

- Inspection and summarization

- Tidy export and codebook generation

---

## Inspired by:

<img src="https://m.media-amazon.com/images/M/MV5BNDNkZDk0ODktYjc0My00MzY4LWE3NzgtNjU5NmMzZDA3YTA1XkEyXkFqcGc@._V1_FMjpg_UX1000_.jpg" alt="Inspiration for Package, Lassie Movie" width="100"/>

## 🧠 L.A.S.S.I.E. Pipeline Summary

| Step | Method           | Purpose                                                                 |
|------|------------------|-------------------------------------------------------------------------|
| L    | `load()`         | Load raw data from a supported source (e.g., MongoDB, UAS, MetricWire). |
| A    | `assure()`       | Validate that required columns exist before processing.                 |
| S    | `score()`        | Apply scoring logic based on predefined or custom rules.                |
| S    | `summarize()`    | Aggregate scored data by participant, session, or custom groups.        |
| I    | `inspect()`      | Visualize distributions or pairwise plots for quality checks.           |
| E    | `export()`       | Save scored and summarized data to tidy files and optionally metadata.  |

---


## 📦 Supported Sources

You may have used M2C2kit tasks via our various integrations, including the ones listed below. Each integration has its own loader class, which is responsible for reading the data and converting it into a format that can be processed by the `m2datakit` package. Keep in mind that you are responsible for ensuring that the data is in the correct format for each loader class.

In the future we anticipate creating loaders for downloading data via API.

| Source Type   | Loader Class          | Key Arguments                            | Notes                                 |
|---------------|------------------------|-------------------------------------------|----------------------------------------|
| `mongodb`     | `MongoDBImporter`      | `source_path` (URL, to JSON)                      | Expects flat or nested JSON documents. |
| `multicsv`    | `MultiCSVImporter`     | `source_map` (dict of CSV paths)          | Each activity type is its own file.    |
| `metricwire`  | `MetricWireImporter`   | `source_path` (glob pattern or default)   | Processes JSON files from unzipped export. |
| `qualtrics`    | `QualtricsImporter`     | `source_path` (URL to CSV)         | Each activity's trial saves data to a new column.    |
| `m2c2kit-api`         | `BackendImporter`     | `source_path` (URL to CSV)         | Each activity's trial saves data to a new column.    |

## 🛠️ **Setup Environment to run the Notebook**

<span style="color:red">**Before running the notebook:**</span>

1. Make sure **Python and Jupyter extensions** are <u>**enabled**</u>. Please follow the M2C2 DataKit Notebook Guide if needed.
2. Run the cells <u>**in order**</u>.
3. Let every block <u>**completely**</u> run before moving on to the next block. This will help you catch errors early and avoid confusion. <span style="color:green">✓</span>

<span style="color:red">**Differences in file paths dependent on your device:**</span>
- Mac example: `"/Users/yourname/Desktop/qualtrics_export.csv"`
- Windows example: `"C:/Users/yourname/Desktop/qualtrics_export.csv"`

Windows users should use forward slashes `/` or double backslashes `\\` in file paths.

#### ***Tip:** Let every block <u>**fully**</u> run before moving on to the next block. This will help you catch errors early and avoid confusion later.* <span style="color:green">✓</span> 

##### To 'Run' a block, go to the left side of the block and hover, then click the play button. 



##### <span style="color:green">**No changes needed here.** Please continue running the next code block.</span> (This may take awhile)

In [1]:
import sys
import subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "m2c2-datakit"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "--force-reinstall", "--no-cache-dir", "m2datakit", "python-dotenv"], check=True)

CompletedProcess(args=['C:\\Users\\alwer\\AppData\\Local\\Microsoft\\WindowsApps\\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\\python.exe', '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--no-cache-dir', 'm2datakit', 'python-dotenv'], returncode=0)

## **Check the Version of Your Scoring Package** 
#### (this information becomes useful to help troubleshoot if you receive any errors)


##### <span style="color:green">**No changes needed here.** Please continue running the next code block.</span>

In [2]:
import os
import re
import glob
import json
from dotenv import load_dotenv
import m2c2_datakit as m2c2
m2c2.core.get_package_version()

'0.1.107'

##### <span style="color:green">**No changes needed here.** Please continue running the next code block.</span>
##### **Troubleshooting:** If you get an error, please <u>'Restart'</u> the Jupyter kernel and rerun the code block below.

In [ ]:
from pathlib import Path

output_folder = Path("tidy_m2c2_export")
output_folder.mkdir(parents=True, exist_ok=True)

# Activity name  ->  summarize function.  Trim this to the tasks in your dataset,
# or add your own custom summarize callables.
summary_func_map = {
    "Symbol Search": m2c2.tasks.symbol_search.summarize,
    "Grid Memory": m2c2.tasks.grid_memory.summarize,
    "Color Dots": m2c2.tasks.color_dots.summarize,
    "Shopping List": m2c2.tasks.shopping_list.summarize,
    "Trailmaking": m2c2.tasks.trailmaking.summarize,
    "Go No Go": m2c2.tasks.go_no_go.summarize,
    "Color Shapes": m2c2.tasks.color_shapes.summarize,
    "Color Squares": m2c2.tasks.color_squares.summarize,
    "Go No Go Fade": m2c2.tasks.go_no_go_fade.summarize,
    "Color Match": m2c2.tasks.color_match.summarize,
    "Stroop": m2c2.tasks.stroop.summarize,
    "Digit Span": m2c2.tasks.digit_span.summarize,
    "Odd or Even": m2c2.tasks.even_odd.summarize,
    "JOLO": m2c2.tasks.jolo.summarize,
    "Implicit Association Test": m2c2.tasks.iat.summarize,
    "Face Naming Task": m2c2.tasks.facename.summarize,
    "Symbol-Number Matching Task": m2c2.tasks.symbol_number_matching.summarize,
    "Digit Symbol": m2c2.tasks.symbol_number_matching.summarize,
}

# Note: ^ this also means that a user could specify their own summarize functions as needed!

---
---
---

## <u><span style="color:yellow">**Platform: Metricwire**</span></u>

**Continue here if you are using MetricWire data downloaded from the MetricWire dashboard. If you are not, please skip to the next platform.**

### **Set your MetricWire file path**

> **⚠️ WARNING:** This is a very important note.
> Make sure to read this before proceeding!

1. **Your downloaded MetricWire *data MUST be unzipped* to work.**

2. *<span style="color:red; font-weight:bold;">**Keep the end of the file path exactly as shown below:**</span> **`/*/*/*/*/*.json`*** <br><br>

   **Why?** Each `/*` represents one folder level in your unzipped MetricWire export. The `*` acts as a wildcard, meaning it will match any folder name at that level. The final `*.json` tells Python to find all JSON files within those folders.<br><br>

   *<span style="color:red; font-weight:bold;">**<u>Troubleshooting:</u> Folder structures may vary slightly. If this path produces an error or does not find your JSON files, remove one `/*` segment and rerun the code. The path should then end with:**</span> **`/*/*/*/*.json`***


3. Please ensure the dashes in the path are forward slashes (/) and not backslashes (\).  
4. When copying your file path, make sure to remove any extra spaces or quotes that may have been added inadvertently.


<u>For example:</u>

```python
source_path_mw = "C:/Users/yourname/Downloads/metricwire/unzipped/*/*/*/*/*.json"
```

In [ ]:
# IMPORTANT - PLEASE READ THE ABOVE BEFORE PROCEEDING!!!!!!

source_path_mw = "PATH/TO/YOUR/metricwire/unzipped/*/*/*/*/*.json" # 👈 edit file location to point at YOUR unzipped MetricWire export. This is the ONLY code you need to edit. 
# example: source_path_mw = "/Users/username/Downloads/metricwire_export/*/*/*/*/*.json" 
# IMPORTANT: The path must point to your unzipped MetricWire export and end with the wildcard folder pattern shown. 
# If the files are not detected or you receive an error, see the troubleshooting note above.

mw = m2c2.core.pipeline.LASSIE().load(source_name="metricwire", source_path=source_path_mw)
mw

# Data from Metricwire
mw = m2c2.core.pipeline.LASSIE().load(source_name="metricwire", source_path=source_path_mw)
mw.assure(required_columns=m2c2.core.config.settings.STANDARD_GROUPING_FOR_AGGREGATION_METRICWIRE)
mw.score()

##### <span style="color:green">**No changes needed here.** Please continue running the next code block.</span>
##### <mark>**Your export will be located in a folder called 'tidy_m2c2_export' and will be in the same folder as your Jupyter notebook.**</mark>

In [ ]:
mw.summarize(summary_func_map = summary_func_map, groupby_cols=m2c2.core.config.settings.STANDARD_GROUPING_FOR_AGGREGATION_METRICWIRE)
# mw.inspect()
mw.export(file_basename="export_metricwire", directory=output_folder)
mw.export_codebook(filename="codebook_metricwire.md", directory=output_folder)

##### *This is the end of your MetricWire M2C2 data processing*

---
---
---






## <u><span style="color:yellow">**Platform: Qualtrics</span>**</u>

### Set your Qualtrics file path

> **⚠️ WARNING:** This is a very important note.
> Make sure to read this before proceeding!

1. Please ensure the dashes in the path are forward slashes (/) and not backslashes (\).  
2. When copying your file path, make sure to remove any extra spaces or quotes that may have been added inadvertently.

In [ ]:
# 👇 Edit this to point at YOUR Qualtrics export.
qualtrics = m2c2.core.pipeline.LASSIE().load(
    source_name="qualtrics",
    source_path="PATH/TO/YOUR/qualtrics_export.csv",  # 👈 edit this
)

# Please ensure the dashes in the path are forward slashes (/) and not backslashes (\).  
# If you are on Windows, you can use double backslashes (\\) instead of single backslashes (\) to avoid escape character issues.

##### <span style="color:green">**No changes needed here.** Please continue running the next code block.</span>
##### <mark>**Your export will be located in a folder called 'tidy_m2c2_export' and will be in the same folder as your Jupyter notebook.**</mark>

In [ ]:
# Data from Qualtrics
qualtrics.assure(required_columns=['ResponseId'])
qualtrics.score()
qualtrics.summarize(summary_func_map = summary_func_map, groupby_cols=['ResponseId'])
#qualtrics.inspect()
qualtrics.export(file_basename="export_qualtrics", directory=output_folder)
qualtrics.export_codebook(filename="codebook_qualtrics.md", directory=output_folder)

##### *This is the end of your Qualtrics M2C2 data processing*

## <span style="color:red">**STOP HERE IF YOU USED METRICWIRE OR QUALTRICS**</span>

---
---
---

## Platform: MongoDB

In [ ]:
# Define source_folder relative to current working directory
#source_folder_mdb = os.path.abspath(os.path.join(os.pardir, "datakit/data/production-mongo-export"))
#source_path_mdb = f"{source_folder_mdb}/data_exported_120424_1010am.json"

source_path_mdb = "~/Desktop/tmb_dev_data_snapshot_100925.json"

# Data from demo M2C2 study on PSU production server
mdb = m2c2.core.pipeline.LASSIE().load(source_name="mongodb", source_path=source_path_mdb)
mdb.assure(required_columns=[
        "study_uid",
        "user_uid",
        "activity_name",
    ])

In [ ]:
mdb.whats_inside()

In [ ]:
mdb.score()
mdb.inspect()

In [ ]:
mdb.summarize(summary_func_map = summary_func_map, groupby_cols=['user_uid'])

In [ ]:
mdb.export(file_basename="MDB", directory="tidy", formats=[".csv", "SQL", "RData"])

## Platform: Multiple CSVs of Trial-level data

In [ ]:
# Data from REBOOT Study (UCF and PSU) was manually merged so we have two csvs to load
source_map = {
    "Symbol Search": "~/Documents/GitHub/datakit/data/reboot/m2c2kit_manualmerge_symbol_search_all_ts-20250402_151939.csv",
    "Grid Memory": "~/Documents/GitHub/datakit/data/reboot/m2c2kit_manualmerge_grid_memory_all_ts-20250402_151940.csv"
}

# Data from REBOOT Study (UCF and PSU) was manually merged so we have two csvs to load
mcsv = m2c2.core.pipeline.LASSIE().load(source_name="multicsv", source_map=source_map)
mcsv.assure(required_columns=['participant_id'])
mcsv.score()
mcsv.summarize(summary_func_map = summary_func_map)
mcsv.inspect()
mcsv.export(file_basename="export_multicsv", directory=output_folder)
mcsv.export_codebook(filename="codebook_multicsv.md", directory=output_folder)

# see whats inside
mcsv.whats_inside()

In [ ]:
umass = m2c2.core.pipeline.LASSIE().load(
    source_name="m2c2-static",
    source_path="../data/m2c2_results_UMass.csv",
)

umass.assure(required_columns=umass.recommended_groupby_cols)
umass.score()
umass.flat_scored.head()

from scipy.constants import u

summary_func_map = {
    "Symbol Search": m2c2.tasks.symbol_search.summarize,
    "Grid Memory": m2c2.tasks.grid_memory.summarize,
    "Color Dots": m2c2.tasks.color_dots.summarize,
    "Shopping List": m2c2.tasks.shopping_list.summarize,
    "Trailmaking": m2c2.tasks.trailmaking.summarize,
    "Go No Go": m2c2.tasks.go_no_go.summarize,
    "Color Shapes": m2c2.tasks.color_shapes.summarize,
    "Color Squares": m2c2.tasks.color_squares.summarize,
    "Go No Go Fade": m2c2.tasks.go_no_go_fade.summarize,
    "Color Match": m2c2.tasks.color_match.summarize,
    "Stroop": m2c2.tasks.stroop.summarize,
    "Digit Span": m2c2.tasks.digit_span.summarize,
    "Odd or Even": m2c2.tasks.even_odd.summarize,
    "JOLO": m2c2.tasks.jolo.summarize,
    "Implicit Association Test": m2c2.tasks.iat.summarize,
    "Face Naming Task": m2c2.tasks.facename.summarize,
    "Symbol-Number Matching Task": m2c2.tasks.symbol_number_matching.summarize,
    "Digit Symbol": m2c2.tasks.symbol_number_matching.summarize,
}
umass.summarize(summary_func_map=summary_func_map)

## Platform: M2C2 Production Backend API

This code block might not work. This is an experimental feature right now.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

m2c2api_user = os.getenv("m2c2api_username")
m2c2api_pass = os.getenv("m2c2api_password")

m2c2_api = m2c2.core.pipeline.LASSIE().load(
    source_name="m2c2kit-api",
    source_path="https://api.m2c2kit.com",
    study_id="M2C2_TestNewBackend",
    username=m2c2api_user,
    password=m2c2api_pass,
    payload={"limit": 10},
    pipeline_name="all_uploaded_data",
)
m2c2_api.score()
m2c2_api.summarize(summary_func_map = summary_func_map)
m2c2_api.export(file_basename="export_multicsv", directory=output_folder)